# 🔍 CLAHE — Realce de Contraste para Inspeção de Obras

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Pré-processamento de Imagens

---

## Contexto

Imagens fotográficas de obras frequentemente sofrem de **iluminação irregular**: regiões muito claras (reflexo do sol em superfícies de concreto) ao lado de regiões muito escuras (sombras de vigas, cantos, cofres de forma). Isso dificulta tanto a inspeção humana quanto algoritmos de detecção automática de patologias.

A equalização global de histograma — a técnica mais simples — melhora o contraste geral, mas **amplifica ruído** em regiões homogêneas e **apaga detalhes** em áreas de transição.

O **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) resolve isso dividindo a imagem em blocos locais (*tiles*) e equalizando cada bloco separadamente, com um limite de amplificação (`clipLimit`) que evita o estouro de ruído.

### Por que isso importa no contexto BIM?

Ao confrontar fotografias de obra com o modelo IFC, a qualidade do pré-processamento determina diretamente a confiabilidade da detecção de anomalias (trincas, segregações, desalinhamentos). Um CLAHE bem calibrado é muitas vezes a diferença entre um algoritmo que funciona e um que gera falsos positivos.

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Testar o ambiente com imagem sintética (sem precisar do Drive) |
| 2 | Montar o Google Drive e configurar caminhos |
| 3 | Processar todas as imagens da pasta em lote |
| 4 | Visualizar comparativos e histogramas |
| 5 | Calibrar parâmetros com grade visual |

> 💡 **Recomendação:** execute a Seção 1 primeiro. Ela não depende de nenhum arquivo externo e confirma que seu ambiente está funcionando corretamente.

---
## Seção 1 — Teste com imagem sintética

Antes de conectar ao Drive, vamos validar que o pipeline funciona. Criamos aqui uma imagem sintética que simula as condições típicas de uma foto de obra:

- **Gradiente de iluminação** — simula luz solar incidindo de um lado
- **Região escura** — simula sombra de fôrma ou viga
- **Textura de ruído** — simula grão fotográfico
- **Risco fino** — simula uma trinca superficial

Se o CLAHE funcionar corretamente, o risco deve ficar **mais visível** após o processamento, mesmo nas regiões escuras.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ── Criar imagem sintética 512×512
h, w = 512, 512
img_sintetica = np.zeros((h, w), dtype=np.uint8)

# Gradiente horizontal (simula iluminação desigual)
gradiente = np.tile(np.linspace(30, 200, w, dtype=np.uint8), (h, 1))
img_sintetica = gradiente.copy()

# Região escura no canto inferior esquerdo (simula sombra)
img_sintetica[320:, :200] = (img_sintetica[320:, :200] * 0.3).astype(np.uint8)

# Ruído gaussiano
ruido = np.random.normal(0, 8, (h, w)).astype(np.int16)
img_sintetica = np.clip(img_sintetica.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# Trinca simulada (linha diagonal fina, tom escuro)
for i in range(100, 400):
    j = int(150 + (i - 100) * 0.6)
    if 0 <= j < w:
        img_sintetica[i, j:j+2] = max(0, int(img_sintetica[i, j]) - 60)

# ── Aplicar CLAHE
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img_sintetica)

# ── Visualizar
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

eixos[0].imshow(img_sintetica, cmap='gray', vmin=0, vmax=255)
eixos[0].set_title('Imagem sintética — Original\n(gradiente + sombra + trinca + ruído)', fontsize=11)
eixos[0].axis('off')

eixos[1].imshow(img_clahe, cmap='gray', vmin=0, vmax=255)
eixos[1].set_title('Após CLAHE  (clipLimit=3.0, tileGrid=8×8)\n↑ trinca mais visível, sombra corrigida', fontsize=11)
eixos[1].axis('off')

plt.suptitle('✅ Teste de ambiente — imagem sintética', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Ambiente OK — pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar caminhos

A célula abaixo abre uma janela de autorização do Google. Clique no link, selecione sua conta e cole o código de autorização.

> ⚠️ **Atenção:** a pasta `fotos_obra` precisa estar em *Meu Drive* ou ter um **atalho** adicionado ao seu Drive.  
> Se a pasta for compartilhada externamente: *Drive → clique direito na pasta → Organizar → Adicionar atalho ao Drive*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
import os
from pathlib import Path

# ── Caminhos — ajuste PASTA_ENTRADA se sua estrutura for diferente
PASTA_ENTRADA = Path('/content/drive/MyDrive/fotos_obra')
PASTA_SAIDA   = Path('/content/drive/MyDrive/fotos_obra_clahe')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Parâmetros CLAHE (altere aqui para toda a pipeline)
CLIP_LIMIT = 3.0
TILE_GRID  = (8, 8)

# ── Extensões aceitas
EXTENSOES = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')

# ── Verificar se a pasta existe
if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
    print('   Verifique o caminho ou adicione um atalho conforme instrução acima.')
else:
    arquivos = sorted([p for p in PASTA_ENTRADA.iterdir() if p.suffix.lower() in EXTENSOES])
    print(f'📂 Entrada : {PASTA_ENTRADA}')
    print(f'📁 Saída   : {PASTA_SAIDA}')
    print(f'⚙️  clipLimit={CLIP_LIMIT}, tileGrid={TILE_GRID}')
    print(f'🖼️  {len(arquivos)} imagem(ns) encontrada(s)')

---
## Seção 3 — Processamento em lote

### Como funciona o pipeline

```
Imagem colorida (BGR)
        │
        ▼
  Escala de cinza          ← cv2.cvtColor(img, COLOR_BGR2GRAY)
        │
        ▼
  Divide em tiles (blocos) ← tileGridSize=(8,8) → 64 blocos locais
        │
        ▼
  Equaliza cada bloco      ← histograma local, não global
        │
        ▼
  Corta amplificação       ← clipLimit=3.0 → redistribui excesso
        │
        ▼
  Interpola bordas         ← bilinear, evita artefatos de bloco
        │
        ▼
  Imagem CLAHE resultante
```

Cada imagem processada é salva na pasta de saída com o prefixo `clahe_`, preservando o nome original.

In [ ]:
from tqdm.notebook import tqdm

def aplicar_clahe(caminho: Path, clip: float, grid: tuple) -> tuple:
    """Lê imagem → cinza → CLAHE. Retorna (gray, clahe, nome)."""
    bgr  = cv2.imread(str(caminho))
    if bgr is None:
        raise ValueError(f'Não foi possível abrir: {caminho.name}')
    gray  = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid)
    return gray, clahe.apply(gray), caminho.name


erros = []

for arq in tqdm(arquivos, desc='Processando imagens'):
    try:
        _, img_clahe, nome = aplicar_clahe(arq, CLIP_LIMIT, TILE_GRID)
        cv2.imwrite(str(PASTA_SAIDA / f'clahe_{nome}'), img_clahe)
    except Exception as e:
        erros.append((arq.name, str(e)))

print(f'\n✅ {len(arquivos) - len(erros)} arquivo(s) salvo(s) em: {PASTA_SAIDA}')
if erros:
    print('\n⚠️  Arquivos com erro:')
    for nome, msg in erros:
        print(f'   {nome}: {msg}')

---
## Seção 4 — Visualização comparativa e histogramas

### 4a. Painel Original × CLAHE

O painel abaixo mostra lado a lado as primeiras imagens da pasta. Observe especialmente:
- Regiões de **sombra**: devem ficar mais legíveis
- **Trincas e texturas finas**: devem ganhar contraste local
- **Áreas homogêneas** (céu, parede lisa): não devem ser excessivamente granuladas — se estiverem, reduza o `clipLimit`

In [ ]:
MAX_EXIBIR = 6  # ajuste conforme necessário
amostra = arquivos[:MAX_EXIBIR]
n = len(amostra)

fig, eixos = plt.subplots(n, 2, figsize=(12, 4.5 * n))
if n == 1:
    eixos = [eixos]

for i, arq in enumerate(amostra):
    gray, gray_clahe, nome = aplicar_clahe(arq, CLIP_LIMIT, TILE_GRID)

    eixos[i][0].imshow(gray, cmap='gray')
    eixos[i][0].set_title(f'Original — {nome}', fontsize=9)
    eixos[i][0].axis('off')

    eixos[i][1].imshow(gray_clahe, cmap='gray')
    eixos[i][1].set_title(f'CLAHE  clip={CLIP_LIMIT} grid={TILE_GRID} — {nome}', fontsize=9)
    eixos[i][1].axis('off')

plt.suptitle('Painel Comparativo — Inspeção de Obras', fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()

caminho_painel = '/content/painel_comparativo.png'
plt.savefig(caminho_painel, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Painel salvo em {caminho_painel}')

### 4b. Histogramas — interpretação

O histograma mostra a **distribuição de intensidades** dos pixels (0 = preto, 255 = branco).

- **Antes do CLAHE:** histograma concentrado em uma faixa estreita → baixo contraste
- **Após o CLAHE:** histograma mais espalhado → contraste local aumentado

> Um histograma perfeitamente plano seria a equalização global ideal (impossível na prática). O CLAHE busca um equilíbrio entre espalhamento e controle de ruído.

In [ ]:
if arquivos:
    gray, gray_clahe, nome = aplicar_clahe(arquivos[0], CLIP_LIMIT, TILE_GRID)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.hist(gray.ravel(), bins=256, range=(0, 255), color='steelblue', alpha=0.85)
    ax1.set_title(f'Histograma — Original\n{nome}', fontsize=10)
    ax1.set_xlabel('Intensidade (0–255)')
    ax1.set_ylabel('Nº de pixels')
    ax1.set_xlim(0, 255)

    ax2.hist(gray_clahe.ravel(), bins=256, range=(0, 255), color='darkorange', alpha=0.85)
    ax2.set_title(f'Histograma — Após CLAHE\n{nome}', fontsize=10)
    ax2.set_xlabel('Intensidade (0–255)')
    ax2.set_xlim(0, 255)

    plt.tight_layout()
    plt.show()

---
## Seção 5 — Calibração de parâmetros

### Entendendo os parâmetros

| Parâmetro | O que controla | Efeito de aumentar |
|-----------|----------------|--------------------|
| `clipLimit` | Teto de amplificação do histograma local | Mais contraste, mais ruído |
| `tileGridSize` | Tamanho do bloco local (em pixels) | Valores menores = mais local e detalhado |

### Regras práticas para inspeção de obras

- **Trincas finas em concreto:** `clipLimit=2–3`, `tileGrid=(8,8)` — realça bordas sem estourar ruído
- **Iluminação muito irregular (obra a céu aberto):** `clipLimit=4–5`, `tileGrid=(16,16)` — corrige gradientes amplos
- **Imagem já bem iluminada:** `clipLimit=1–2` — evita amplificar artefatos desnecessariamente

A grade abaixo permite comparar visualmente todas as combinações para a primeira imagem da sua pasta.

In [ ]:
if arquivos:
    img_ref = cv2.cvtColor(cv2.imread(str(arquivos[0])), cv2.COLOR_BGR2GRAY)
    nome_ref = arquivos[0].name

    clips = [1.0, 2.0, 3.0, 5.0]
    grids = [4, 8, 16]

    n_linhas = len(clips)
    n_colunas = len(grids) + 1  # +1 para coluna 'Original'

    fig, eixos = plt.subplots(n_linhas, n_colunas,
                               figsize=(4.2 * n_colunas, 3.8 * n_linhas))

    for i, clip in enumerate(clips):
        # Coluna 0 = referência
        eixos[i][0].imshow(img_ref, cmap='gray')
        eixos[i][0].set_ylabel(f'clipLimit = {clip}', fontsize=10, fontweight='bold')
        eixos[i][0].set_title('Original' if i == 0 else '', fontsize=9)
        eixos[i][0].axis('off')

        for j, g in enumerate(grids):
            c   = cv2.createCLAHE(clipLimit=clip, tileGridSize=(g, g))
            res = c.apply(img_ref)
            eixos[i][j + 1].imshow(res, cmap='gray')
            eixos[i][j + 1].set_title(f'grid = ({g},{g})' if i == 0 else '', fontsize=9)
            eixos[i][j + 1].axis('off')

    plt.suptitle(f'Grade de parâmetros CLAHE\n{nome_ref}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('👆 Compare as combinações e escolha os valores para CLIP_LIMIT e TILE_GRID na Seção 2.')

---
## Próximos passos

Com as imagens pré-processadas na pasta `fotos_obra_clahe`, você está pronto para o próximo módulo:

```
fotos_obra_clahe/
        │
        ├── Detecção de bordas (Canny, Sobel)
        ├── Segmentação de patologias (U-Net, SAM)
        └── Confronto com modelo IFC → registro BCF
```

### Referências

- Zuiderveld, K. (1994). *Contrast Limited Adaptive Histogram Equalization*. Graphics Gems IV, Academic Press.
- OpenCV docs: [`cv2.createCLAHE`](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html)
- ABNT NBR 6118:2014 — Projeto de estruturas de concreto (referência para classificação de patologias)
